# Домашнее задание 7. Сборка конвейера CI/CD
Если у вас еще нет аккаунта в GitLab, вам нужно будет его создать:
1. Перейдите на [GitLab](https://gitlab.com/) и войдите в свой аккаунт.
2. Нажмите на кнопку New Project (Новый проект).
3. Выберите Create blank project (Создать пустой проект).
4. Укажите имя проекта и описание (по желанию).
5. Выберите уровень видимости проекта (Public).
6. Нажмите Create project (Создать проект).
7. Дополните файл .gitlab-ci.yml необходимыми джобами и отправьте в репозиторий.

## 1. Настроить CI/CD-пайплайн для ML-сервиса с использованием GitLab




Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Вам дан рабочий код пайплайна и черновик файла .gitlab-ci.yml. Перепишите yaml в [ячейке](#scrollTo=s55MrS66JXWs)


*Ожидаемый артефакт: список коммитов в [ячейке](#scrollTo=gErasBmRSHjb) и ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=F0uQqbe3iHqE)*    

### Проверяем работоспособность пайплайна

### Проверка статуса пайплайна

После настройки файла `.gitlab-ci.yml`, вы можете закоммитить изменения и запушить их в репозиторий.

GitLab автоматически запустит пайплайн, и вы сможете наблюдать за его выполнением в разделе CI/CD своего проекта.

Что нужно сделать:

1. Перейдите в свой проект на GitLab.
2. Нажмите на вкладку CI/CD и выберите Pipelines.
3. Вы увидите список запущенных пайплайнов. Нажмите на последний, чтобы увидеть выполнение.
4. Убедитесь, что все джобы выполнены успешно (отмечены зеленым цветом).
5. Приложите ссылку на статус выполнения в разделе Pipelines **своего** репозитория на GitLab.

**Решение**

Для выполнения задания выбоан аналог среды GitLab - GitHub Action

**Список коммитов** - https://github.com/RuiGoNk/mlops-hw7-cicd/commits/main/


**Ссылка на выполненный пайплайн** - https://github.com/RuiGoNk/mlops-hw7-cicd/actions/runs/25657162595

## 2. Обосновать стратегию деплоя (развертывания, Blue-Green, Canary, Rolling, Shadow) и оценить влияние на риски




Изучите [инструмент](https://github.com/npryce/adr-tools) для учета архитектурных решений и запишите **причины**, по которым мы начали использовать стратегию деплоя и **риски**, к которым нас привело такое решение.



*Ожидаемый артефакт: архитектурное решение в формате ADR в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

**Решение**

## ADR-001: Выбор стратегии деплоя для ML-сервиса

### Статус
Принято

### Контекст
У нас есть ML-сервис (модель RandomForestClassifier на датасете Iris), который нужно обновлять без простоя. Нам нужна стратегия развертывания новой версии модели.

### Рассмотренные альтернативы
1. **Blue-Green** — два окружения, переключение всех пользователей разом
2. **Canary** — постепенное переключение (10% → 50% → 100%)
3. **Rolling** — постепенная замена инстансов

### Решение
Выбрана стратегия **Blue-Green Deployment**

### Обоснование
- Проще в реализации, чем Canary
- Быстрый откат при ошибках (секунды вместо минут)
- Для учебного проекта избыточная сложность Canary не нужна

### Последствия
- **Плюсы:** Нет простоя при обновлении, мгновенный откат
- **Минусы:** Нужно вдвое больше ресурсов (два окружения)
- **Риски:** При переключении возможна кратковременная потеря сессий

## 3. Реализовать стратегию развертывания

Реализуйте стратегию, выбранную на предыдущем [шаге](#scrollTo=hoQdM6SrJXXE).



*Ожидаемый артефакт: yaml в текстовой [ячейке](#scrollTo=hycprahZcUrJ)*

**Решение**

https://github.com/RuiGoNk/mlops-hw7-cicd/blob/main/docker-compose.blue.yml

https://github.com/RuiGoNk/mlops-hw7-cicd/blob/main/docker-compose.green.yml

https://github.com/RuiGoNk/mlops-hw7-cicd/blob/main/docker-compose.proxy.yml

## 4. Спланировать A/B-тестирование для ML-модели

Вспомните материалы [семинара](https://colab.research.google.com/drive/1TM1yieSFhUqVxBferzbcexpAtK00lGYe?usp=sharing) и опишите параметры эксперимента.



*Ожидаемый артефакт: код в [ячейке](#scrollTo=OluzjqEhaIpM)*

In [11]:
import random
import time
from datetime import datetime

AB_TEST_CONFIG = {
    'group_a_percent': 50,
    'group_b_percent': 50,
    'duration_days': 7,
    'min_requests': 1000,
    'metrics': {
        'accuracy_threshold': 0.95,
        'latency_threshold_ms': 100
    }
}

test_results = {
    'group_a': {'requests': 0, 'success': 0, 'total_latency': 0},
    'group_b': {'requests': 0, 'success': 0, 'total_latency': 0}
}

def get_model_version(user_id):
    """Определяет, какая модель достанется пользователю"""
    hash_value = hash(str(user_id)) % 100
    if hash_value < AB_TEST_CONFIG['group_a_percent']:
        return 'blue_v1.0.0'
    else:
        return 'green_v1.1.0'

def log_prediction(version, is_correct, latency_ms):
    """Логирует результат предсказания"""
    group = 'group_a' if version == 'blue_v1.0.0' else 'group_b'
    test_results[group]['requests'] += 1
    test_results[group]['success'] += 1 if is_correct else 0
    test_results[group]['total_latency'] += latency_ms

def get_current_stats():
    """Возвращает текущую статистику теста"""
    stats = {}
    for group, data in test_results.items():
        if data['requests'] > 0:
            accuracy = data['success'] / data['requests']
            avg_latency = data['total_latency'] / data['requests']
        else:
            accuracy = 0
            avg_latency = 0
        stats[group] = {
            'requests': data['requests'],
            'accuracy': accuracy,
            'avg_latency_ms': avg_latency
        }
    return stats

def should_stop_test():
    """Проверяет, нужно ли остановить тест"""
    stats = get_current_stats()
    total_requests = stats['group_a']['requests'] + stats['group_b']['requests']

    if total_requests < AB_TEST_CONFIG['min_requests']:
        return False

    group_b_acc = stats['group_b']['accuracy']
    group_a_acc = stats['group_a']['accuracy']

    if group_b_acc < AB_TEST_CONFIG['metrics']['accuracy_threshold']:
        print(f"Новая модель показывает точность {group_b_acc:.3f} < {AB_TEST_CONFIG['metrics']['accuracy_threshold']}")
        return True

    if group_b_acc < group_a_acc - 0.05:
        print(f"Новая модель хуже старой ({group_b_acc:.3f} vs {group_a_acc:.3f})")
        return True

    return False

def decide_winner():
    """Принимает решение по итогам теста"""
    stats = get_current_stats()
    group_a_acc = stats['group_a']['accuracy']
    group_b_acc = stats['group_b']['accuracy']

    print("ИТОГИ A/B ТЕСТИРОВАНИЯ")
    print(f"Старая модель (Blue v1.0.0): точность {group_a_acc:.3f}")
    print(f"Новая модель (Green v1.1.0): точность {group_b_acc:.3f}")

print("ЗАПУСК A/B ТЕСТИРОВАНИЯ")

users = [f"user_{i}" for i in range(100)]
for i, user in enumerate(users[:50]):
    version = get_model_version(user)
    if version == 'green_v1.1.0':
        is_correct = random.random() < 0.95
        latency = random.uniform(20, 80)
    else:
        is_correct = random.random() < 0.92
        latency = random.uniform(25, 95)

    log_prediction(version, is_correct, latency)

stats = get_current_stats()
print(f"Статистика:")
for group, data in stats.items():
    print(f"   {group}: {data['requests']} запросов, точность {data['accuracy']:.3f}, задержка {data['avg_latency_ms']:.1f} мс")

should_stop = should_stop_test()
winner = decide_winner()

ЗАПУСК A/B ТЕСТИРОВАНИЯ
Статистика:
   group_a: 24 запросов, точность 0.917, задержка 53.7 мс
   group_b: 26 запросов, точность 1.000, задержка 49.1 мс
ИТОГИ A/B ТЕСТИРОВАНИЯ
Старая модель (Blue v1.0.0): точность 0.917
Новая модель (Green v1.1.0): точность 1.000


## 5. Создать CI/CD-пайплайн для ML-сервиса с использованием GitHub Actions



*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*



Вам нужно вспомнить, какие части ML-проекта вы будете сохранять, чтобы получить воспроизводимый пайплайн.

Проверяем работоспособность пайплайна

Вам дан рабочий код пайплайна и черновик файла ci.yml. Используйте GitHub Actions и перепишите [шаг](#scrollTo=NGcDFbCFJXV_) name: Make pipeline reproducible

Копируем ci.yml в правильную директорию .github/workflows

Начинаем отправку в репозиторий

После настройки workflow каждый раз при пуше в репозиторий GitHub Actions будет автоматически запускать конвейер. Пожалуйста, приложите ссылку на статус выполнения в разделе Actions **своего** репозитория на GitHub.


*Ожидаемый артефакт: ссылка на выполненный пайплайн в репозитории в [ячейке](#scrollTo=CQG_D73seauF)*

**Решение**

https://github.com/RuiGoNk/mlops-hw7-cicd/actions/runs/25657664338

## 6. Итоговое оформление

В итоговых выводах дайте 5–8 предложений о своем опыте работы с инструментами модуля: что оказалось простым, что вызвало трудности, какие выводы сделали по обоснованию стратегии деплоя.



## Итоговые выводы

1. **CI/CD (GitHub Actions)** — настроить оказалось проще, чем ожидала. Достаточно создать файл .yml в папке .github/workflows, и пайплайн запускается автоматически при каждом push.

2. **Что вызвало трудности** — синтаксис YAML. Несколько раз пайплайн падал из-за лишних пробелов или неправильных отступов. Также пришлось разбираться с версиями действий (actions/checkout@v4 и т.д.).

3. **Стратегия деплоя** — выбрала Blue-Green, потому что она проще в реализации, чем Canary, и даёт быстрый откат при ошибках. Для учебного проекта этого достаточно.

4. **Главный вывод** — CI/CD и A/B тестирование обязательны для production-сервисов. Без них невозможно гарантировать, что новая модель не сломает пользовательский опыт.